In [2]:
import sys
from pathlib import Path

_root = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "config.py").exists())
if str(_root) not in sys.path:
    sys.path.insert(0, str(_root))

# --- standard library -----------------------------------------------------
import json
import logging
from collections import Counter

# --- third party ----------------------------------------------------------
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

# --- project --------------------------------------------------------------
import config
from src.data import fetch_poems, generate, splits
from src.data import filter as data_filter  # `filter` alone shadows the builtin
from src.plots import figures
from src.eval import format_check, grounding, judge, swap_test

%load_ext autoreload
%autoreload 2

# One logging setup for every notebook. Also quietens the HTTP and hub loggers
# that would otherwise bury this project's own output — see config for why each
# suppression is safe.
config.configure_logging()

# 3 — Judge validation

**This notebook is a gate, and it runs before any GPU time is spent.**

Everything downstream rests on one assumption: that an LLM judge can tell an
interpretation written *for* a poem from one written for a different poem. If
it cannot, then every score it produces for the five arms is noise wearing a
number, and no amount of training or evaluation repairs that.

So the assumption is tested first, on the easiest possible case — **teacher
interpretations**, which the funnel has already verified quote their poem
correctly. These are known-grounded text. A judge that cannot separate matched
from mismatched *here* has no chance on a 0.5B model's output.

The design is the swap test. One interpretation, three poems:

| Condition | Poem shown | Role |
|---|---|---|
| `matched` | the poem it was written for | — |
| `mismatched_random` | a random poem by a **different** author | standard control |
| `mismatched_same_author` | a different poem by the **same** author | strict control |

Two gaps follow, and their difference is the finding:

- **grounding gap** = matched − mismatched_random
- **poem-level gap** = matched − mismatched_same_author
- **author component** = the difference between them

The third row is why the strict control exists. An author's themes recur, so
text that merely sounds like Dickinson will beat a random Whitman poem while
scoring no better on one Dickinson poem than another. The standard control
alone would call that well grounded.

**This is a relative measurement, which is why it needs no ground truth.** The
judge's scale is never calibrated — only kept consistent. Miscalibration moves
every condition together and cancels in the difference.

## Building the pairs

Every constraint below fails *silently*. Nothing raises if the strict control
draws a near-duplicate, or if the standard control happens to draw the same
author — the run completes and reports a plausible number that means something
else. `swap_test.check_all` is what turns those into errors instead of
findings.

In [3]:
corpus, _ = data_filter.build_corpus(fetch_poems.load_cached(),
                                    generate.load_cached())

# PoetryDB publishes some poems under several titles. Drawing one as the
# "different poem by the same author" would turn the strict condition into the
# matched condition, and the poem-level gap would collapse to zero for a reason
# that has nothing to do with the model.
duplicates = data_filter.near_duplicate_ids(corpus)

# The same 150 evaluation poems the arms will later be generated for, drawn
# with the same seed — so this validation and the real run judge the same poems.
exemplars = splits.reserve_exemplars(corpus, n=config.N_FEWSHOT, seed=config.SEED)
folds = splits.make_folds(corpus, config.N_FOLDS,
                          group_key=config.FOLD_GROUP_KEY,
                          seed=config.SEED, exclude=exemplars)
eval_set = splits.sample_eval_poems(folds, config.EVAL_PER_FOLD, config.SEED)

pairs = swap_test.build_pairs(
    [{"poem_id": p["poem_id"], "interpretation": p["interpretation"]}
     for p in eval_set],
    corpus, arm="teacher", near_duplicates=duplicates)

swap_test.check_all(pairs, corpus, near_duplicates=duplicates)
print(f"{len(pairs)} pairs — {len(pairs) // len(config.SWAP_CONDITIONS)} "
      f"interpretations x {len(config.SWAP_CONDITIONS)} conditions")
print(f"{len(pairs) * len(config.JUDGES)} calls if both judges score everything")

INFO dropped 75 duplicate records
INFO corpus: 2536 poems from 3081 raw
INFO excluding 3 group(s) reserved as exemplars: Anne Bronte, Robert Louis Stevenson, Sir Thomas Wyatt
INFO 5 folds, sizes [497, 497, 496, 496, 496] (from 124 authors)
INFO built 450 pairs for arm 'teacher' (150 interpretations x 3 conditions)
INFO swap-test pair construction passed every structural check


450 pairs — 150 interpretations x 3 conditions
900 calls if both judges score everything


## One pair, read by eye first

Before trusting 450 automated scores, read one. The three rows below are the
same interpretation judged against three different poems — if the design is
sound, the first should score high and the other two low.

In [4]:
by_id = {p["poem_id"]: p for p in corpus}
example = [p for p in pairs if p.poem_id == pairs[0].poem_id]

print(f'INTERPRETATION written for: "{by_id[example[0].poem_id]["title"]}" '
      f'by {by_id[example[0].poem_id]["author"]}\n')
print(example[0].interpretation[:400], "...\n")
print("scored against:")
for pair in example:
    shown = by_id[pair.shown_id]
    print(f'  {pair.condition:<24} "{shown["title"][:34]}" by {shown["author"]}')

INTERPRETATION written for: "Epitaph. on James Craggs, Esq. in Westminster Abbey." by Alexander Pope

1. Central idea - The poem is a eulogy for James Craggs, celebrating his integrity, loyalty, and moral self-sufficiency, while mourning his untimely death at thirty-five. It contrasts his inner worth with the fleeting nature of titles and public envy.

2. Key images -  
- “Who broke no promise, served no private end” – an image of unwavering ethical conduct in public life.  
- “Ennobled by himself ...

scored against:
  matched                  "Epitaph. on James Craggs, Esq. in " by Alexander Pope
  mismatched_random        "On Retirement" by Philip Freneau
  mismatched_same_author   "Autumn." by Alexander Pope


## Scoring

**Resumable and cached.** Every score is appended to
`results/judge_scores_<judge>.jsonl` the moment it arrives, and already-scored
pairs are skipped on restart — so an interrupted run loses nothing and
re-running this cell costs nothing. That file is committed with the repo, which
is how these results survive the session.

Scores are keyed by judge name in the **filename as well as in every record**,
so the two judges cannot be pooled even by accident. `assert_single_judge`
raises on a mixed set rather than quietly averaging across two instruments.

In [5]:
scored = {}
for spec in config.JUDGES:
    try:
        scored[spec.name] = judge.score_all(pairs, corpus, spec)
    except Exception as error:
        print(f"{spec.name}: {type(error).__name__}: {str(error)[:200]}")

for name, records in scored.items():
    usable = [r for r in records if r.get("score") is not None]
    print(f"{name:<14} {len(usable)}/{len(pairs)} scored, "
          f"{len(records) - len(usable)} unparseable")

INFO gpt4o_mini: 450 cached, 0 to score
INFO gemini_flash: 199 cached, 251 to score


gemini_flash:   0%|          | 0/251 [00:00<?, ?pair/s]

gpt4o_mini     450/450 scored, 0 unparseable
gemini_flash   450/450 scored, 0 unparseable


### Replies the judge never resolved

A pair whose reply cannot be parsed is recorded with `score=None` and excluded
from every mean — **never defaulted to a number**. A malformed reply scored as
5 would be indistinguishable from a genuine middling verdict, and it would drag
every gap toward zero in a way nobody could see in the aggregate.

Excluding them is not free either, so the ones that fail are listed rather than
counted. Two things decide whether the exclusion is safe:

**Are they concentrated in one condition?** Loss spread evenly across
conditions cancels in a difference. Loss concentrated in `matched` does not —
it removes observations from the condition that anchors both gaps.

**Are they reproducible?** A reply lost to truncation on one draw and returned
on the next is a transmission fault, and re-running recovers it. A pair that
fails every time is a property of the judge on that input, and no number of
retries will produce a verdict.

The distinction matters because retrying until a reply parses is not neutral:
a reasoning judge that overruns its token budget succeeds on the attempts where
it happened to reason *less*, so retry-until-parsed quietly selects the
short-reasoning draws.

In [6]:
by_id = {p["poem_id"]: p for p in corpus}

for name, records in scored.items():
    unresolved = [r for r in records if r.get("score") is None]
    print(f"{name}: {len(unresolved)}/{len(records)} unresolved "
          f"({len(unresolved) / len(records):.2%})")
    if unresolved:
        counts = Counter(r["condition"] for r in unresolved)
        print(f"   by condition: {dict(counts)}")
    for r in unresolved:
        wrote_for = by_id[r["poem_id"]]
        print(f"\n   poem {r['poem_id']}  {wrote_for['author']} — "
              f"{wrote_for['title'][:44]}")
        print(f"   condition {r['condition']}")
        print(f"   reply     {r['reply']!r}")
    print()

gpt4o_mini: 0/450 unresolved (0.00%)

gemini_flash: 0/450 unresolved (0.00%)



**What the surviving failure is.** The reply above shows the judge
quoting the poem's own lines back to itself, checking each quotation in turn,
and running out of budget before reaching a verdict. That pair was re-scored
repeatedly and failed every time — so it is not truncation bad luck, it is this
judge reliably reasoning past its budget on this input.

It is left unscored and reported. The alternative — raising the token budget
until it resolves — would change the judge's configuration partway through a
run and make its scores incomparable with the rest, which is a worse problem
than one missing observation.

**The primary judge has none of this.** GPT-4o-mini answers with a bare integer
and resolves every pair, because it does not reason before answering. The
failure mode belongs to the secondary judge alone, and every hypothesis test is
computed from the primary — so no headline number is affected. It is reported
here because it is a real, measured property of the instrument, and because a
reader comparing the two judges deserves to know that one of them declines to
answer occasionally.

## The gate

`MIN_JUDGE_SEPARATION` is stated as an effect size, not a p-value. With 150
paired observations almost any non-zero gap is statistically significant; the
question here is whether the instrument discriminates *usefully*, and one point
on a ten-point scale is the least that could be called separation.

**A failure here is not a bad result — it is a stop sign.** It would mean the
judge cannot tell grounded from ungrounded on text known to be grounded, and
the design has to change before any GPU time is spent.

In [7]:
for name, records in scored.items():
    if not records:
        continue
    print(f"--- {name} ---")
    for condition, mean in judge.condition_means(records).items():
        print(f"  {condition:<24} {mean:.2f}")
    gaps = judge.gaps(records)
    print(f"  {'grounding gap':<24} {gaps['grounding_gap']:+.2f}   "
          f"(matched - random)")
    print(f"  {'poem-level gap':<24} {gaps['poem_level_gap']:+.2f}   "
          f"(matched - same author)")
    print(f"  {'author component':<24} {gaps['author_component']:+.2f}   "
          f"(the difference)")
    passed, verdict = judge.separation_verdict(records)
    print(f"\n  {verdict}\n")

--- gpt4o_mini ---
  matched                  8.97
  mismatched_random        1.07
  mismatched_same_author   1.17
  grounding gap            +7.90   (matched - random)
  poem-level gap           +7.80   (matched - same author)
  author component         +0.10   (the difference)

  PASS — matched scores 7.90 above mismatched_random (threshold 1.0)

--- gemini_flash ---
  matched                  9.99
  mismatched_random        1.00
  mismatched_same_author   1.01
  grounding gap            +8.99   (matched - random)
  poem-level gap           +8.98   (matched - same author)
  author component         +0.01   (the difference)

  PASS — matched scores 8.99 above mismatched_random (threshold 1.0)



## Do the judges have room to discriminate?

A mean hides saturation. If every pair in a condition scored the identical
value, the variance is zero and any gap computed from it is a **floor or
ceiling effect** rather than a measurement — the number could not have come out
otherwise.

This matters most for the author component, which is a difference between two
controls. A control pinned to the floor makes that difference structurally
zero, and reporting it as *"no author effect was found"* would state a property
of the scale as though it were a property of the model.

In [8]:
for name, records in scored.items():
    print(f"--- {name} ---")
    for condition, stats in judge.score_spread(records).items():
        flag = "  <- SATURATED" if stats["saturated"] else ""
        print(f"  {condition:<24} n={stats['n']:<4} "
              f"range {stats['min']}-{stats['max']:<3} "
              f"distinct {stats['distinct']:<3} {stats['counts']}{flag}")
    saturated = judge.saturated_conditions(records)
    if saturated:
        print(f"  ! {name} returned ONE value for: {', '.join(saturated)}")
        print(f"    any gap involving these is a floor effect, not a measurement")
    print()

--- gpt4o_mini ---
  matched                  n=150  range 8-10  distinct 3   {8: 6, 9: 143, 10: 1}
  mismatched_random        n=150  range 1-3   distinct 3   {1: 143, 2: 4, 3: 3}
  mismatched_same_author   n=150  range 1-6   distinct 5   {1: 136, 2: 7, 3: 5, 4: 1, 6: 1}

--- gemini_flash ---
  matched                  n=150  range 9-10  distinct 2   {9: 1, 10: 149}
  mismatched_random        n=150  range 1-1   distinct 1   {1: 150}  <- SATURATED
  mismatched_same_author   n=150  range 1-2   distinct 2   {1: 148, 2: 2}
  ! gemini_flash returned ONE value for: mismatched_random
    any gap involving these is a floor effect, not a measurement



**Read the ranges, not the means.** The judge that looks more decisive is
not necessarily the better instrument — a judge that answers only "1" or "10"
has told you the pairing is right or wrong and nothing else, while one that
uses the middle of the scale can express degrees of mismatch.

**The forward-looking risk.** Both judges were validated on *teacher* text:
fluent, accurate, quoting correctly. Notice where each puts it on the scale.
Model outputs will be worse than the teacher and will land in the middle —
precisely where these distributions are thinnest.

The gate proves the judges can tell **right from wrong**. It does not prove
they can tell **better from worse**, and ranking five arms needs the second.
Two things guard against it later: the `template` arm is a built-in probe, since
a generic interpretation should land mid-scale and would reveal a binary judge
if it does not; and the pairwise win rates avoid the problem entirely, because
comparing two outputs directly needs no calibrated absolute scale.

## Do the two judges agree?

The one place the judges legitimately meet — and they are still not pooled.
The scores stay in separate arguments and separate columns, and what is
computed is agreement *between* them rather than an average *of* them.

**Raw agreement is reported alongside Cohen's κ, never alone.** Raw agreement
is inflated whenever one verdict dominates, and here it does: most control
pairs sit at the floor, so two judges agreeing by both saying "1" is barely
evidence of anything. κ discounts exactly that chance agreement.

**κ comes back undefined (NaN) when neither judge varies.** That is the correct
answer rather than a bug — with no variance there is no chance agreement to
correct for. The NaN is itself the finding: the judges agree completely on a
question neither was able to answer in more than one way.

Zheng et al. 2023 report >80% judge–human agreement on MT-Bench. This is
judge–judge rather than judge–human, and on an easier task, so it is a related
reference point rather than a like-for-like comparison.

In [9]:
primary = scored.get(config.PRIMARY_JUDGE.name)
secondary = scored.get(config.SECONDARY_JUDGE.name)

if primary and secondary:
    agreement = judge.inter_judge_agreement(primary, secondary)
    a, b = agreement["judges"]
    print(f"{a} vs {b}   on {agreement['n']} pairs scored by both\n")
    print(f"  exact same score        {agreement['exact_agreement']:.1%}")
    print(f"  same side of the scale  {agreement['same_side']:.1%}")
    print(f"  Cohen's kappa           {agreement['cohens_kappa']:.3f}")
    print(f"  mean difference         {agreement['mean_difference']:+.2f}  "
          f"({a} minus {b})")
    print(f"\n  MT-Bench reference: >80% judge-human agreement "
          f"(Zheng et al. 2023) — different task, different comparison")

gpt4o_mini vs gemini_flash   on 450 pairs scored by both

  exact same score        62.7%
  same side of the scale  99.8%
  Cohen's kappa           0.995
  mean difference         -0.27  (gpt4o_mini minus gemini_flash)

  MT-Bench reference: >80% judge-human agreement (Zheng et al. 2023) — different task, different comparison


## Is the judge reading, or matching strings?

The gate is passed. This asks whether it was passed **for the right reason** —
and it is the check that decides whether an LLM judge is worth its cost here at
all.

The output schema demands two or three exact quotations. So a `matched` pair
contains literal substrings of the poem shown, and a `mismatched` pair contains
none. A judge could therefore separate the two perfectly by string matching,
which `src.eval.grounding` already does for free, deterministically, with no
API and no judge bias. If that is what is happening, the gap is real and the
judge is redundant.

The ablation removes the shortcut: every quoted span is replaced with a
placeholder, and **nothing else changes** — same poems, same controls, same
seed, same prompt. One variable. Whatever separation survives is the judge
responding to the interpretation's *claims* rather than to its quotations.

Stripped pairs carry the arm name `teacher_noquotes`, so they can never
overwrite the real scores or be silently mixed with them.

In [10]:
stripped_pairs = swap_test.build_pairs(
    [{"poem_id": p["poem_id"], "interpretation": p["interpretation"]}
     for p in eval_set],
    corpus, arm="teacher", near_duplicates=duplicates, strip_quotes=True)
swap_test.check_all(stripped_pairs, corpus, near_duplicates=duplicates)

print("what the judge now sees instead of a quotation:\n")
print(stripped_pairs[0].interpretation[:400], "...")

INFO built 450 pairs for arm 'teacher_noquotes' (150 interpretations x 3 conditions)
INFO swap-test pair construction passed every structural check


what the judge now sees instead of a quotation:

1. Central idea - The poem is a eulogy for James Craggs, celebrating his integrity, loyalty, and moral self-sufficiency, while mourning his untimely death at thirty-five. It contrasts his inner worth with the fleeting nature of titles and public envy.

2. Key images -  
- [quotation removed] – an image of unwavering ethical conduct in public life.  
- [quotation removed] – a portrait of self-made  ...


In [11]:
stripped = {}
for spec in config.JUDGES:
    try:
        stripped[spec.name] = judge.score_all(stripped_pairs, corpus, spec)
    except Exception as error:
        print(f"{spec.name}: {type(error).__name__}: {str(error)[:200]}")

INFO gpt4o_mini: 450 cached, 0 to score
INFO gemini_flash: 450 cached, 0 to score


In [12]:
for name in scored:
    if name not in stripped:
        continue
    quoted_gaps = judge.gaps(scored[name])
    strip_gaps = judge.gaps(stripped[name])
    print(f"--- {name} ---")
    print(f"  {'':<24}{'quoted':>9}{'stripped':>10}{'retained':>10}")
    for key in ("grounding_gap", "poem_level_gap", "author_component"):
        kept = (f"{strip_gaps[key] / quoted_gaps[key]:.0%}"
                if quoted_gaps[key] else "-")
        print(f"  {key:<24}{quoted_gaps[key]:>+9.2f}"
              f"{strip_gaps[key]:>+10.2f}{kept:>10}")
    print()

--- gpt4o_mini ---
                             quoted  stripped  retained
  grounding_gap               +7.90     +7.03       89%
  poem_level_gap              +7.80     +6.39       82%
  author_component            +0.10     +0.64      640%

--- gemini_flash ---
                             quoted  stripped  retained
  grounding_gap               +8.99     +8.94       99%
  poem_level_gap              +8.98     +8.85       99%
  author_component            +0.01     +0.09      700%



**How to read this.**

*The gap largely survives* → the judge is responding to interpretive claims,
not to quotation. It is doing work the free substring checker cannot, and it
earns its cost.

*The gap collapses* → it was string matching. The honest conclusion would then
be that `grounding.py` is the better instrument — faster, deterministic, free,
and immune to judge bias — and that the LLM judge added nothing. That would be
a finding worth reporting, not a failure.

**Watch the author component especially.** With quotations present, they
dominate: a Whitman poem and a different Dickinson poem both fail to contain
the quoted lines, so both controls score at the floor and the two look
identical. Remove the quotations and the same-author control can rise, because
author-flavoured prose fits a same-author poem better than a stranger's.

If the author component only becomes visible here, that is the empirical
justification for the strict swap condition — and it means the quoted
measurement **understates** how much apparent grounding is author recognition.

**This carries forward to the arms.** They will quote too, since the schema
demands it, so their gaps will carry the same inflation and mask the same
author effect. The no-quotes condition therefore belongs in the evaluation of
every arm, not only in this validation.

## Saving the result

Written to `results/swap_test_summary.csv`, one **row per judge** — never a
pooled mean, which would answer a question nobody asked. Generated rather than
hand-typed, so the numbers in the report cannot drift from the scores they came
from.

Two artifacts persist beyond this session, both committed:

| File | What it holds |
|---|---|
| `results/judge_scores_<judge>.jsonl` | every individual score, resumable |
| `results/swap_test_summary.csv` | the aggregate, one row per judge |

In [13]:
summary = judge.save_summary([r for r in scored.values() if r])
summary

INFO wrote /Users/adelinchaushev/Desktop/PoetryIntepretations./results/swap_test_summary.csv


,judge,judge_model,arm,n_pairs,n_scored,n_unparseable,mean_matched,mean_mismatched_random,mean_mismatched_same_author,grounding_gap,poem_level_gap,author_component,n_paired_poems
0,gpt4o_mini,gpt-4o-mini,teacher,450,450,0,8.967,1.067,1.167,7.900,7.80,0.100,150
1,gemini_flash,gemini-3.5-flash,teacher,450,450,0,9.993,1.000,1.013,8.993,8.98,0.013,150


## What this does and does not establish

**Establishes:** whether each judge separates matched from mismatched on
known-grounded text, and how much of that separation survives the strict
same-author control.

**Does not establish:** that the judge agrees with human readers. Zheng et al.
2023 report >80% judge–human agreement on MT-Bench, but that is their benchmark
and their task; no human annotation exists for poetry grounding here, and none
is claimed. What is measured is agreement with a *known-correct pairing*, which
is a weaker but genuinely different check.

**The author component is the number to carry forward.** If it is large, then
much of what looks like grounding is the judge recognising an author rather
than reading a poem — and the poem-level gap, not the grounding gap, is the
defensible measurement for every arm that follows.